<a href="https://colab.research.google.com/github/ArinzeIhematulam/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArinzeIhematulam/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Logistic Regression first, then Random Forest — both scored via predict_proba at Precision@K, since this is a ranking ("which pages first?") problem, not a bare classification one. Following the training-honest-models guide's table for "yes/no with an observed label," I start with the readable option (Logistic Regression) and move to a stronger one (Random Forest) only if it earns the extra complexity on the same metric — Precision@10/20/50 — as my Week-4 baseline (the ctr_below_tier_average rule). Both models get compared against the baseline on the exact same held-out data, not different slices, since a fair comparison is the whole point of this notebook.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import duckdb, pandas as pd
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}')")

base = con.sql("""
    WITH filtered AS (
        SELECT *, EXTRACT(day FROM report_date) AS day_of_month
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    h1 AS (
        SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
               SUM(gsc_impressions) AS impressions_h1, SUM(gsc_clicks) AS clicks_h1,
               AVG(gsc_avg_position) AS avg_position_h1,
               COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_h1,
               STDDEV(gsc_avg_position) AS position_std_h1
        FROM filtered WHERE day_of_month <= 15 GROUP BY content_hash_id
    ),
    h2 AS (
        SELECT content_hash_id, SUM(gsc_clicks) AS clicks_h2
        FROM filtered WHERE day_of_month > 15 GROUP BY content_hash_id
    )
    SELECT h1.*, h2.clicks_h2
    FROM h1 JOIN h2 USING (content_hash_id)
    WHERE h1.clicks_h1 > 0
""").df()

base["ctr_h1"] = base["clicks_h1"] / base["impressions_h1"]
base["declining"] = (base["clicks_h2"] < base["clicks_h1"]).astype(int)

pos_bins = [0, 3, 10, 20, 50, 100000]
pos_labels = ["top_3", "4-10", "11-20", "21-50", "50+"]
base["position_tier_h1"] = pd.cut(base["avg_position_h1"], bins=pos_bins, labels=pos_labels)
tier_avg_ctr = base.groupby("position_tier_h1", observed=True)["ctr_h1"].transform("mean")
base["ctr_gap_h1"] = tier_avg_ctr - base["ctr_h1"]
base["baseline_score"] = (base["ctr_gap_h1"] * base["impressions_h1"]).clip(lower=0)

print(f"Total rows: {len(base)}, unique clients: {base['client_hash_id'].nunique()}")

# Grouped 80/20 split by client
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(base, groups=base["client_hash_id"]))
train_df, test_df = base.iloc[train_idx], base.iloc[test_idx]

print(f"Train: {len(train_df)} rows, {train_df['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test_df)} rows, {test_df['client_hash_id'].nunique()} clients")
print(f"Test base rate (declining): {test_df['declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 51586, unique clients: 41
Train: 44583 rows, 32 clients
Test:  7003 rows, 9 clients
Test base rate (declining): 0.585


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouped by client (client_hash_id), not by row. Same reasoning as ML-01's holdout experiment: a random row-level split could let the same client's pages appear in both train and test, letting a model "learn" client identity rather than the general pattern. I split clients 80/20, so no client appears in both sets.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.